# Traffic Sign Detection with YOLO: find + classify 🚦🔍
In the last project, the signs were already **cut out** for us. A real car camera sees the **whole road**. So now we need to **find** the signs first.

**Dataset:** GTSDB (German Traffic Sign *Detection* Benchmark): 900 road images (1360×800) with a box around every sign.

**Two-stage system** (like real driver-assistance systems):
1. **YOLO detector** → *where* are the signs? (1 class: "sign")
2. **GTSRB classifier** from the last project → *which* sign is it? (43 classes)

Why not let YOLO learn all 43 classes? GTSDB has only ~850 signs for training, so many classes would have just 1-5 examples. Our classifier saw 26,000 sign crops. **Use each model for what it's good at.**

| Part | What | Topic |
|---|---|---|
| 1 | Setup | Python |
| 2 | Explore GTSDB: how small are the signs? | Computer Vision |
| 3 | Convert the labels to YOLO format | CV / data |
| 4 | Train YOLO (transfer learning from COCO) | Deep learning |
| 5 | Evaluate the detector (mAP, precision, recall, small signs) | ML theory |
| 6 | Two-stage pipeline: detect → crop → classify | CV |
| 7 | Deployment: ONNX + a standalone app | Deployment |

# Part 1 – Setup

In [ ]:
from pathlib import Path
import json
import random
import shutil
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
from ultralytics import YOLO

# AMD GPU (ROCm) fix: MIOpen fails to compile its BatchNorm kernel on this PC.
torch.backends.cudnn.enabled = False
device = 0 if torch.cuda.is_available() else "cpu"     # ultralytics wants 0 (= first GPU) or "cpu"
print(f"device: {device}")

random.seed(42)
np.random.seed(42)

GTSDB_DIR = Path("data/gtsdb/FullIJCNN2013")
YOLO_DIR = Path("data/gtsdb/yolo")                     # our converted dataset goes here

# Class names from the classifier project (same 43 class ids as GTSDB!)
CLASS_NAMES = json.load(open("deploy/labels.json"))["class_names"]

# Part 2 – Explore GTSDB
The labels are in `gt.txt`, one line per sign: `image;x1;y1;x2;y2;class`
- `(x1, y1)` = top-left corner of the box, `(x2, y2)` = bottom-right corner, in **pixels**.
- Images with **no sign** do not appear in the file at all.

In [ ]:
gt = pd.read_csv(GTSDB_DIR / "gt.txt", sep=";", header=None,
                 names=["file", "x1", "y1", "x2", "y2", "class_id"])
gt["width"] = gt["x2"] - gt["x1"]
gt["height"] = gt["y2"] - gt["y1"]

all_images = sorted(p.name for p in GTSDB_DIR.glob("*.ppm"))
print(f"images: {len(all_images)}   signs: {len(gt)}   "
      f"images with at least 1 sign: {gt['file'].nunique()}   images with NO sign: {len(all_images) - gt['file'].nunique()}")
gt.head()

In [ ]:
# How BIG are the signs? This decides almost everything for a detector.
plt.figure(figsize=(10, 3.5))
plt.hist(gt["width"], bins=40)
plt.axvline(32, color="red", linestyle="--", label="32 px")
plt.xlabel("sign width (pixels) in a 1360 px wide image")
plt.ylabel("number of signs")
plt.title(f"Sign size: median {gt['width'].median():.0f} px, smallest {gt['width'].min()} px, "
          f"{(gt['width'] < 32).mean() * 100:.0f}% are smaller than 32 px")
plt.legend()
plt.show()

⚠️ **Small objects!** Most signs are only 20–60 pixels wide in a 1360-pixel image (about 2–4% of the width).

If we shrink the image to 640 px (YOLO's default), a 20 px sign becomes **9 px**: just a few dots. That's why we train at **1024 px**: more pixels per sign, but slower.

In [ ]:
def draw_boxes(ax, boxes, labels=None, color="lime"):
    """boxes: list of (x1, y1, x2, y2) in pixels."""
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2))
        if labels is not None:
            ax.text(x1, y1 - 6, labels[i], color="black", fontsize=8,
                    bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"))


busy = gt["file"].value_counts().index[0]           # the image with the most signs
rows = gt[gt["file"] == busy]
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(Image.open(GTSDB_DIR / busy))
draw_boxes(ax, rows[["x1", "y1", "x2", "y2"]].values, [CLASS_NAMES[c] for c in rows["class_id"]])
ax.set_title(f"{busy}: {len(rows)} signs")
ax.axis("off")
plt.show()

# Part 3 – Convert to YOLO format
YOLO wants **one text file per image**, same name, with one line per object:
```
class  x_center  y_center  width  height        <- all divided by the image size (0..1)
0      0.585     0.536     0.030  0.044
```
- **Normalized (0..1)** so the labels stay correct when YOLO resizes the image.
- **Center + size**, not corners like `gt.txt`.
- Class is always `0` ("sign"): the detector only finds signs, the classifier names them.
- Images **without signs** get an **empty** label file. They teach YOLO what is *not* a sign (fewer false alarms).

**Split:** the official one: images 0-599 = train, 600-899 = test. From train we keep 100 images for validation.

In [ ]:
IMG_W, IMG_H = 1360, 800

official_train = [f for f in all_images if int(f[:5]) < 600]
test_files = [f for f in all_images if int(f[:5]) >= 600]
rng = random.Random(42)
val_files = sorted(rng.sample(official_train, 100))
train_files = [f for f in official_train if f not in val_files]
splits = {"train": train_files, "val": val_files, "test": test_files}
print({k: len(v) for k, v in splits.items()})


def to_yolo_line(row):
    """One gt.txt row (corners, pixels) -> one YOLO line (center + size, normalized)."""
    xc = (row.x1 + row.x2) / 2 / IMG_W
    yc = (row.y1 + row.y2) / 2 / IMG_H
    w = (row.x2 - row.x1) / IMG_W
    h = (row.y2 - row.y1) / IMG_H
    return f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"


if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)                          # start clean every time
for split, files in splits.items():
    (YOLO_DIR / "images" / split).mkdir(parents=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True)
    for f in files:
        stem = f[:-4]                                # "00012.ppm" -> "00012"
        # YOLO can't read .ppm -> save as high-quality JPEG
        Image.open(GTSDB_DIR / f).save(YOLO_DIR / "images" / split / f"{stem}.jpg", quality=95)
        lines = [to_yolo_line(r) for r in gt[gt["file"] == f].itertuples()]
        (YOLO_DIR / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))   # empty if no sign

# The dataset description file YOLO reads
data_yaml = YOLO_DIR / "gtsdb.yaml"
data_yaml.write_text("\n".join([
    f"path: {YOLO_DIR.resolve().as_posix()}",     # absolute path to the dataset folder
    "train: images/train",
    "val: images/val",
    "test: images/test",
    "names:",
    "  0: sign",
]) + "\n")
print(data_yaml.read_text())

In [ ]:
# ALWAYS check the conversion: read a label file back and draw it (wrong boxes = wasted training!)
sample = busy[:-4]
split = next(s for s, files in splits.items() if busy in files)
lines = (YOLO_DIR / "labels" / split / f"{sample}.txt").read_text().splitlines()

boxes = []
for line in lines:
    _, xc, yc, w, h = map(float, line.split())
    boxes.append(((xc - w / 2) * IMG_W, (yc - h / 2) * IMG_H, (xc + w / 2) * IMG_W, (yc + h / 2) * IMG_H))

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(Image.open(YOLO_DIR / "images" / split / f"{sample}.jpg"))
draw_boxes(ax, boxes, color="magenta")
ax.set_title("Boxes read back from the YOLO label file (must match the signs)")
ax.axis("off")
plt.show()

# Part 4 – Train YOLO (transfer learning)
We start from **YOLO11n** ("n" = nano, the smallest: ~2.6M parameters), pretrained on **COCO** (80 everyday objects, including "stop sign"). It already knows edges, shapes and colors. We only teach it our task.

Important settings:
- `imgsz=1024`: bigger input → small signs keep more pixels.
- `fliplr=0.5` (left-right flip): **OK this time!** In the classifier project, flips were forbidden because "Keep right" ↔ "Keep left". But the detector only says "sign", and a flipped sign is still a sign. The classifier later gets the *original, unflipped* crop.
- `mosaic` (YOLO default): glues 4 images into 1 → more signs per image, more variety.

In [ ]:
model = YOLO("yolo11n.pt")      # COCO-pretrained weights (downloaded automatically, ~5 MB)

start = time.time()
model.train(
    data=str(data_yaml),
    imgsz=1024,
    epochs=50,
    batch=4,                      # small batch: bigger ones crash this AMD laptop GPU (Windows GPU timeout)
    workers=4,
    fliplr=0.5,
    patience=20,                  # stop early if validation mAP doesn't improve for 20 epochs
    seed=42,
    project=str(Path("runs/gtsdb").resolve()),
    name="yolo11n_signs",
    exist_ok=True,
    device=device,
    verbose=False,
)
print(f"training time: {(time.time() - start) / 60:.1f} min")

In [ ]:
run_dir = Path("runs/gtsdb/yolo11n_signs")
best_weights = run_dir / "weights" / "best.pt"

# YOLO saves its own plots. The most useful one: loss and mAP over the epochs.
fig, ax = plt.subplots(figsize=(15, 7.5))
ax.imshow(Image.open(run_dir / "results.png"))
ax.axis("off")
plt.show()

# Part 5 – Evaluate the detector on the TEST set
- **Precision:** of all the boxes YOLO draws, how many are real signs? (low = false alarms)
- **Recall:** of all the real signs, how many did YOLO find? (low = missed signs, dangerous!)
- **mAP50:** average precision when a box counts as correct if it overlaps the real box by ≥ 50% (**IoU** ≥ 0.5).
- **mAP50-95:** the same, averaged over stricter overlaps (50%…95%). Measures how *exact* the boxes are.

**IoU** (Intersection over Union) = overlap area ÷ total area of both boxes. 1 = perfect, 0 = no overlap.

In [ ]:
detector = YOLO(best_weights)
metrics = detector.val(data=str(data_yaml), split="test", imgsz=1024, device=device, verbose=False,
                       project=str(Path("runs/gtsdb").resolve()), name="test_eval", exist_ok=True)
print(f"precision: {metrics.box.mp:.3f}")
print(f"recall:    {metrics.box.mr:.3f}")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")

In [ ]:
def iou(box, boxes):
    """IoU of one box [4] with many boxes [N, 4]. Boxes are (x1, y1, x2, y2)."""
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda b: (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (area(box) + area(boxes) - inter)


CONF = 0.25     # boxes below this confidence are thrown away

# Run the detector on every test image once, and keep the results
detections = {}                                   # file -> (boxes [N, 4], confidences [N])
for r in detector.predict([str(YOLO_DIR / "images" / "test" / f"{f[:-4]}.jpg") for f in test_files],
                          imgsz=1024, conf=CONF, device="cpu", stream=True, verbose=False):
    # device="cpu": on this AMD laptop GPU, the GPU is sometimes left in a broken state after
    # training ("unspecified launch failure"). The CPU is just as fast here (~0.2 s per image).
    name = Path(r.path).stem + ".ppm"
    detections[name] = (r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy())

# Which real signs were found? (matched to a detection with IoU >= 0.5)
test_gt = gt[gt["file"].isin(test_files)].copy()
found = []
for row in test_gt.itertuples():
    det_boxes, _ = detections[row.file]
    real = np.array([row.x1, row.y1, row.x2, row.y2], dtype=float)
    found.append(len(det_boxes) > 0 and iou(real, det_boxes).max() >= 0.5)
test_gt["found"] = found

# Recall by sign size: are small signs harder?
bins = [0, 24, 32, 48, 64, 200]
test_gt["size"] = pd.cut(test_gt["width"], bins=bins, labels=["<24 px", "24-32", "32-48", "48-64", ">64 px"])
by_size = test_gt.groupby("size", observed=True)["found"].agg(["mean", "count"])
by_size["mean"] *= 100
by_size.columns = ["recall (%)", "signs"]
print(by_size.round(1))
print(f"\noverall recall at conf {CONF}: {test_gt['found'].mean() * 100:.1f}%")

# Part 6 – Two-stage pipeline: detect → crop → classify
For every box YOLO finds:
1. **Crop** it from the *original* full-resolution image, with a **10% margin** around it. Why? The GTSRB training images also show a small border around each sign. The crop should look like what the classifier saw in training!
2. **Classify** the crop with our INT8 ONNX classifier from the last project (`deploy/`).

**End-to-end correct** = the box overlaps a real sign (IoU ≥ 0.5) **and** the class is right.

⚠️ **Be careful with a perfect score:** GTSRB (the classifier's training data) and GTSDB come from the same research group and similar drives through Germany. If some signs appear in both, the classifier has "seen" them before, and its score here is optimistic. A truly independent test would use signs from another country or camera.

In [ ]:
sys.path.insert(0, "deploy")
import predict_sign                        # our classifier from the last project (ONNX INT8)

MARGIN = 0.10


def crop_with_margin(img, box, margin=MARGIN):
    x1, y1, x2, y2 = box
    mx, my = (x2 - x1) * margin, (y2 - y1) * margin
    return img.crop((max(0, x1 - mx), max(0, y1 - my), min(img.width, x2 + mx), min(img.height, y2 + my)))


def classify_detections(file):
    """Classify every detected box of one test image. Returns a list of (box, det_conf, class_id, cls_conf)."""
    img = Image.open(GTSDB_DIR / file).convert("RGB")
    out = []
    for box, conf in zip(*detections[file]):
        class_id, _, cls_conf = predict_sign.classify_image(crop_with_margin(img, box))
        out.append((box, conf, class_id, cls_conf))
    return out


results = {f: classify_detections(f) for f in test_files}

# End-to-end evaluation
tp, n_pred = 0, 0
wrong_class = 0
for f in test_files:
    real = gt[gt["file"] == f]
    real_boxes = real[["x1", "y1", "x2", "y2"]].values.astype(float)
    used = set()
    for box, conf, class_id, _ in sorted(results[f], key=lambda r: -r[1]):    # most confident first
        n_pred += 1
        if len(real_boxes) == 0:
            continue
        ious = iou(box, real_boxes)
        j = int(ious.argmax())
        if ious[j] >= 0.5 and j not in used:
            used.add(j)
            if class_id == real["class_id"].values[j]:
                tp += 1
            else:
                wrong_class += 1

n_real = len(test_gt)
print(f"real signs: {n_real}   detections: {n_pred}")
print(f"found AND correctly classified: {tp}   found but WRONG class: {wrong_class}")
print(f"end-to-end precision: {tp / n_pred * 100:.1f}%   end-to-end recall: {tp / n_real * 100:.1f}%")
print(f"classifier accuracy on the boxes YOLO found: {tp / (tp + wrong_class) * 100:.1f}%")

In [ ]:
# Show the full system on a few test images
show = [f for f in test_files if len(results[f]) >= 2][:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, f in zip(axes.flat, show):
    ax.imshow(Image.open(GTSDB_DIR / f))
    real = gt[gt["file"] == f]
    draw_boxes(ax, real[["x1", "y1", "x2", "y2"]].values, color="white")              # ground truth
    draw_boxes(ax, [r[0] for r in results[f]],
               [f"{CLASS_NAMES[r[2]]} ({r[3]:.0%})" for r in results[f]], color="lime")  # our prediction
    ax.set_title(f"{f}  (white = real box, green = detected + classified)", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Part 7 – Deployment
Same idea as before: export to **ONNX** so the device needs no PyTorch.

- The car camera always gives the **same image size** (1360×800). So we export with a **fixed rectangular input**: 1360×800 scaled to width 1024 → 1024×602 → padded to **1024×608** (YOLO needs multiples of 32). A square 1024×1024 input would waste ~40% of the computation on gray padding.
- The standalone app [`deploy/detect_signs.py`](deploy/detect_signs.py) does everything by hand with numpy: **letterbox** (resize + pad), run YOLO, **decode** the output, **NMS** (remove duplicate boxes), crop, classify, draw. It needs only `onnxruntime`, `numpy` and `Pillow`.

In [ ]:
onnx_file = detector.export(format="onnx", imgsz=[608, 1024], simplify=True, opset=17, dynamic=False)
shutil.copy(onnx_file, "deploy/sign_detector.onnx")
print(f"detector ONNX: {Path('deploy/sign_detector.onnx').stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
import subprocess

# Run the standalone app like on the device (a separate Python process, no PyTorch, no ultralytics)
test_image = GTSDB_DIR / show[0]
out = subprocess.run([sys.executable, "deploy/detect_signs.py", str(test_image), "--out", "deploy/detection_example.jpg"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(Image.open("deploy/detection_example.jpg"))
ax.set_title("Output of deploy/detect_signs.py (ONNX only)")
ax.axis("off")
plt.show()

In [ ]:
# Speed of the full pipeline per camera frame (CPU, ONNX Runtime)
import onnxruntime as ort
sys.path.insert(0, "deploy")
import detect_signs

frames = [Image.open(GTSDB_DIR / f).convert("RGB") for f in test_files[:30]]
for threads in [1, 4]:
    detect_signs.set_threads(threads)
    detect_signs.detect_and_classify(frames[0])                     # warm-up
    start = time.perf_counter()
    for frame in frames:
        detect_signs.detect_and_classify(frame)
    ms = (time.perf_counter() - start) / len(frames) * 1000
    print(f"{threads} CPU thread(s): {ms:6.1f} ms per frame  ->  {1000 / ms:5.1f} frames per second")